<a href="https://colab.research.google.com/github/Ignitesss/nlp_course_yandex/blob/2025/nlp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade transformers datasets accelerate deepspeed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 11.9 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.18.2-py3-none-any.whl size=1763310 sha256=9f5300881eb8eb46c0a098d6c29868c0331f19aada9c73942760e5dc3dc7c714
  Stored in directory: /root/.cache/pip/wheels/69/ad/2e/e03d4739ddc0417efd8a120c2b9e784005aa226037e558c163
Successfully built deepspeed
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets
from tqdm import tqdm
import multiprocessing
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import evaluate

### Data Preparation

In [ ]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

In [ ]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [ ]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [ ]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [ ]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model.to(device)
model.eval()

batch_size = 64
num_workers = multiprocessing.cpu_count()
print(f"Using batch_size: {batch_size}, num_workers: {num_workers}")

val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,  # shuffle для валидации не нужен
    num_workers=num_workers,
    pin_memory=True if device.type == "cuda" else False,
    collate_fn=transformers.default_data_collator,
    prefetch_factor=2 if num_workers > 0 else None
)

Using device: cuda
Using batch_size: 64, num_workers: 2


In [ ]:
def evaluate_model(model, dataloader, device):
    total_correct = 0
    total_samples = 0

    progress_bar = tqdm(dataloader, desc="Evaluating", unit="batch")

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            token_type_ids = batch["token_type_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )

            predictions = torch.argmax(outputs.logits, dim=1)

            correct = (predictions == labels).sum().item()
            total_correct += correct
            total_samples += labels.size(0)

            current_accuracy = total_correct / total_samples
            progress_bar.set_postfix({
                "acc": f"{current_accuracy:.4f}",
                "batch_acc": f"{correct/labels.size(0):.4f}"
            })

    accuracy = total_correct / total_samples
    return accuracy

In [ ]:
accuracy = evaluate_model(model, val_loader, device)
print(f"Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

Evaluating: 100%|██████████| 632/632 [04:38<00:00,  2.27batch/s, acc=0.9084, batch_acc=0.8261]

Validation Accuracy: 0.9084 (90.84%)


In [ ]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions



I chose option A

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="binary")

    return {
        "accuracy": accuracy,
        "f1": f1,
    }


def quick_evaluate(model, dataset, num_samples=1000):
    model.eval()
    correct = 0
    total = 0

    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    with torch.no_grad():
        for idx in indices:
            example = eval_dataset[int(idx)]
            input_ids = example["input_ids"].unsqueeze(0).to(device)
            attention_mask = example["attention_mask"].unsqueeze(0).to(device)
            label = example["labels"].item()

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            prediction = torch.argmax(outputs.logits, dim=1).item()

            if prediction == label:
                correct += 1
            total += 1

    return correct / total

In [ ]:
# DeBERTa-v3-baseimport torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading QQP dataset...")
qqp = datasets.load_dataset("SetFit/qqp")
print(f"Dataset loaded. Train: {len(qqp['train'])}, Validation: {len(qqp['validation'])}")

model_name = "microsoft/deberta-v3-base"
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)

print(f"Model config: {model.config.model_type}")
print(f"Tokenizer has sep_token: {tokenizer.sep_token}")
print(f"Tokenizer has cls_token: {tokenizer.cls_token}")

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    result["labels"] = examples["label"]
    return result

print("Preprocessing dataset...")
tokenized_qqp = qqp.map(preprocess_function, batched=True)
tokenized_qqp = tokenized_qqp.remove_columns(["text1", "text2", "label"])
tokenized_qqp.set_format("torch")

train_dataset = tokenized_qqp["train"]
eval_dataset = tokenized_qqp["validation"]

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

# Аргументы для обучения
training_args = TrainingArguments(
    output_dir="./deberta-v3-qqp-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    fp16=True if device.type == "cuda" else False,  # Используем mixed precision на GPU
    gradient_accumulation_steps=2,
    warmup_steps=500,
    report_to="none",
)

# Создаем тренер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

Using device: cuda
Loading QQP dataset...


Repo card metadata block was not found. Setting CardData to empty.


Dataset loaded. Train: 363846, Validation: 40430
Loading model: microsoft/deberta-v3-base


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

...

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

Train dataset size: 363846
Eval dataset size: 40430


/tmp/ipython-input-295630605.py:67: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("Starting training...")
train_result = trainer.train()
trainer.save_model()
tokenizer.save_pretrained("./deberta-v3-qqp-finetuned")

print("\nTraining results:")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training completed in {train_result.metrics['train_runtime']:.2f} seconds")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.220000,0.213659,0.911600,0.882007
2,0.158000,0.210178,0.917982,0.891229
3,0.115900,0.241314,0.921766,0.895545



Training results:
Training loss: 0.1829
Training completed in 12584.93 seconds


In [ ]:
print("\nEvaluating on validation set...")
eval_results = trainer.evaluate()
print(f"Validation accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Validation F1-score: {eval_results['eval_f1']:.4f}")
print(f"Validation loss: {eval_results['eval_loss']:.4f}")


Evaluating on validation set...


Validation accuracy: 0.9218
Validation F1-score: 0.8955
Validation loss: 0.2413


In [ ]:
# Сохранение лучшей модели с дополнительной информацией
import json
import os

save_dir = "./deberta-v3-qqp-best"
os.makedirs(save_dir, exist_ok=True)
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

metadata = {
    "model_name": model_name,
    "dataset": "QQP",
    "validation_accuracy": float(eval_results['eval_accuracy']),
    "validation_f1": float(eval_results['eval_f1']),
    "training_args": {
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "epochs": training_args.num_train_epochs,
        "weight_decay": training_args.weight_decay,
    }
}

with open(f"{save_dir}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nModel saved to: {save_dir}")
print(f"Final validation accuracy: {eval_results['eval_accuracy']:.4f}")


Model saved to: ./deberta-v3-qqp-best
Final validation accuracy: 0.9218


### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [ ]:
from typing import List, Tuple
import heapq

class DuplicateFinder:
    def __init__(self, model, tokenizer, train_dataset, device, similarity_threshold=0.8):
        """
        Args:
            model: Fine-tuned модель для определения эквивалентности
            tokenizer: Токенизатор модели
            train_dataset: Датасет с вопросами для поиска
            device: Устройство для вычислений (CPU/GPU)
            similarity_threshold: Порог сходства для дубликатов
        """
        self.model = model
        self.tokenizer = tokenizer
        self.train_dataset = train_dataset
        self.device = device
        self.similarity_threshold = similarity_threshold
        self.model.eval()
        self.model.to(device)
        self.prepare_question_database()

    def prepare_question_database(self):
        """Подготовка базы данных вопросов для быстрого поиска"""
        self.question_pairs = []
        self.questions_dict = {}

        # Собираем все уникальные вопросы
        print("Extracting unique questions from training set...")
        all_questions = []

        for i in tqdm(range(min(10000, len(self.train_dataset))), desc="Processing questions"):
            item = qqp["train"][i]
            all_questions.append((item["text1"], item["label"]))
            all_questions.append((item["text2"], item["label"]))

        unique_questions = list(set([q[0] for q in all_questions]))
        print(f"Found {len(unique_questions)} unique questions")

        self.questions_list = unique_questions[:1000]  # 1000 чтобы было быстрее
        print(f"Using {len(self.questions_list)} questions for duplicate search")

    def predict_similarity(self, question1: str, question2: str) -> float:
        """
        Args:
            question1: Первый вопрос
            question2: Второй вопрос

        Returns:
            Вероятность эквивалентности (0-1)
        """
        # Токенизация
        inputs = self.tokenizer(
            question1,
            question2,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            similarity_score = probs[0][1].item()  # Вероятность класса "эквивалентны"

        return similarity_score

    def find_top_duplicates(self, query_question: str, top_k: int = 5) -> List[Tuple[str, float]]:
        """
        Args:
            query_question: Запрос (вопрос для поиска дубликатов)
            top_k: Количество возвращаемых результатов

        Returns:
            Список кортежей - вопрос, оценка схожести
        """
        heap = []

        # Проходим по всем вопросам в базе
        for candidate_question in tqdm(self.questions_list, desc="Searching"):
            if candidate_question == query_question:
                continue  # Пропускаем сам вопрос

            # Вычисляем схожесть
            similarity_score = self.predict_similarity(query_question, candidate_question)

            # Добавляем в heap если достаточно похоже
            if similarity_score >= self.similarity_threshold:
                heapq.heappush(heap, (similarity_score, candidate_question))

                # Держим только top_k элементов
                if len(heap) > top_k:
                    heapq.heappop(heap)

        # Сортируем результаты по убыванию схожести
        results = sorted(heap, key=lambda x: x[0], reverse=True)
        print(f"Found {len(results)} potential duplicates")

        return [(q, score) for score, q in results]

    def find_duplicates_from_dataset(self, query_question: str, top_k: int = 5) -> List[Tuple[str, float, int]]:
        """
        Args:
            query_question: Запрос
            top_k: Количество результатов

        Returns:
            Список кортежей - вопрос, оценка, метка из датасета
        """
        heap = []

        # Ищем в train dataset QQP
        for i in tqdm(range(min(1000, len(qqp["train"]))), desc="Searching QQP pairs"):
            item = qqp["train"][i]
            text1, text2, label = item["text1"], item["text2"], item["label"]

            # Проверяем первый вопрос пары
            if text1 != query_question:
                score1 = self.predict_similarity(query_question, text1)
                heapq.heappush(heap, (score1, text1, label))
                if len(heap) > top_k:
                    heapq.heappop(heap)

            # Проверяем второй вопрос пары
            if text2 != query_question:
                score2 = self.predict_similarity(query_question, text2)
                heapq.heappush(heap, (score2, text2, label))
                if len(heap) > top_k:
                    heapq.heappop(heap)

        results = sorted(heap, key=lambda x: x[0], reverse=True)

        return [(question, score, label) for score, question, label in results]

In [ ]:
duplicate_finder = DuplicateFinder(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    device=device,
    similarity_threshold=0.7
)

# Тестируем на нескольких примерах
test_queries = [
    "How do I learn Python programming?",
    "What is the best way to lose weight?",
    "How to cook pasta?",
    "What are the symptoms of COVID-19?",
    "Best smartphones in 2023?",
    "How to improve my English speaking skills?",
    "What is machine learning?",
    "How to make money online?",
    "What is the capital of France?",
    "How to fix a leaking faucet?"
]

Extracting unique questions from training set...


Processing questions: 100%|██████████| 10000/10000 [00:01<00:00, 5514.76it/s]

Found 19328 unique questions
Using 1000 questions for duplicate search


In [ ]:
def display_results(query, results, source="database"):
    print(f"QUERY: {query}")
    print(f"SOURCE: {source}")

    if not results:
        print("No duplicates found.")
        return

    for i, (question, score, *extra) in enumerate(results, 1):
        label_info = ""
        if len(extra) > 0 and extra[0] is not None:
            label = extra[0]
            label_info = f" [Dataset label: {'Equivalent' if label == 1 else 'Not equivalent'}]"

        print(f"\n{i}. Similarity: {score:.4f}{label_info}")
        print(f"   Question: {question}")

In [ ]:
print("\n" + "="*80)
print("SEARCHING IN QUESTION DATABASE")
print("="*80)

for i, query in enumerate(test_queries[:5], 1):
    print(f"\n\nTest {i}")
    results = duplicate_finder.find_top_duplicates(query, top_k=5)
    display_results(query, results, "question database")


SEARCHING IN QUESTION DATABASE


Test 1


Searching: 100%|██████████| 1000/1000 [00:32<00:00, 30.90it/s]


Found 0 potential duplicates
QUERY: How do I learn Python programming?
SOURCE: question database
No duplicates found.


Test 2


Searching: 100%|██████████| 1000/1000 [00:30<00:00, 32.27it/s]


Found 2 potential duplicates
QUERY: What is the best way to lose weight?
SOURCE: question database

1. Similarity: 0.9912
   Question: How do I suck it up and lose weight?

2. Similarity: 0.9860
   Question: What should I do for weight loss?


Test 3


...

Found 0 potential duplicates
QUERY: How to cook pasta?
SOURCE: question database
No duplicates found.


Test 4


Searching: 100%|██████████| 1000/1000 [00:34<00:00, 29.11it/s]


Found 0 potential duplicates
QUERY: What are the symptoms of COVID-19?
SOURCE: question database
No duplicates found.


Test 5


Searching: 100%|██████████| 1000/1000 [00:32<00:00, 30.91it/s]

Found 0 potential duplicates
QUERY: Best smartphones in 2023?
SOURCE: question database
No duplicates found.


In [ ]:
print("\n\n" + "="*80)
print("SEARCHING IN QQP DATASET PAIRS")
print("="*80)

for i, query in enumerate(test_queries[5:], 1):
    print(f"\n\nTest {i}")
    results = duplicate_finder.find_duplicates_from_dataset(query, top_k=3)
    display_results(query, results, "QQP dataset")



SEARCHING IN QQP DATASET PAIRS


Test 1


Searching QQP pairs: 100%|██████████| 1000/1000 [01:03<00:00, 15.63it/s]


QUERY: How to improve my English speaking skills?
SOURCE: QQP dataset

1. Similarity: 0.9994 [Dataset label: Equivalent]
   Question: How can I improve my English Language?

2. Similarity: 0.9992 [Dataset label: Equivalent]
   Question: How can I improve my English in all aspects?

3. Similarity: 0.9992 [Dataset label: Equivalent]
   Question: How can I improve my communication skills in English?


Test 2


Searching QQP pairs: 100%|██████████| 1000/1000 [01:03<00:00, 15.78it/s]


QUERY: What is machine learning?
SOURCE: QQP dataset

1. Similarity: 0.0482 [Dataset label: Equivalent]
   Question: If more vacuum energy appears with expansion and it has no limit, can infinite of this energy be created? If yes is energy infinite?

2. Similarity: 0.0006 [Dataset label: Equivalent]
   Question: How do mountain ranges form, and what are some of the major mountain ranges in Oklahoma?

3. Similarity: 0.0005 [Dataset label: Equivalent]
   Question: Are fat burning pills helpful along with exercise? What are the best fat burning pills (non-steroidal)?


Test 3


...

QUERY: How to make money online?
SOURCE: QQP dataset

1. Similarity: 0.9993 [Dataset label: Equivalent]
   Question: What are ways of earning money online?

2. Similarity: 0.9993 [Dataset label: Equivalent]
   Question: What are ways of earning money online?

3. Similarity: 0.9992 [Dataset label: Equivalent]
   Question: What is an easy way make money online?


Test 4


Searching QQP pairs: 100%|██████████| 1000/1000 [01:01<00:00, 16.28it/s]


QUERY: What is the capital of France?
SOURCE: QQP dataset

1. Similarity: 0.0032 [Dataset label: Equivalent]
   Question: If more vacuum energy appears with expansion and it has no limit, can infinite of this energy be created? If yes is energy infinite?

2. Similarity: 0.0004 [Dataset label: Equivalent]
   Question: Are fat burning pills helpful along with exercise? What are the best fat burning pills (non-steroidal)?

3. Similarity: 0.0003 [Dataset label: Equivalent]
   Question: How do mountain ranges form, and what are some of the major mountain ranges in Oklahoma?


Test 5


Searching QQP pairs: 100%|██████████| 1000/1000 [01:01<00:00, 16.16it/s]

QUERY: How to fix a leaking faucet?
SOURCE: QQP dataset

1. Similarity: 0.0004 [Dataset label: Equivalent]
   Question: If more vacuum energy appears with expansion and it has no limit, can infinite of this energy be created? If yes is energy infinite?

2. Similarity: 0.0004 [Dataset label: Equivalent]
   Question: How do mountain ranges form, and what are some of the major mountain ranges in Oklahoma?

3. Similarity: 0.0003 [Dataset label: Equivalent]
   Question: Are fat burning pills helpful along with exercise? What are the best fat burning pills (non-steroidal)?


In [ ]:
print("\n\n" + "="*80)
print("VERIFICATION WITH KNOWN QQP DUPLICATES")
print("="*80)

# Берем несколько примеров из QQP, которые помечены как дубликаты
known_duplicates = []
for i in range(min(1000, len(qqp["train"]))):
    item = qqp["train"][i]
    if item["label"] == 1:  # Эквивалентные вопросы
        known_duplicates.append((item["text1"], item["text2"]))
        if len(known_duplicates) >= 5:
            break

print(f"Found {len(known_duplicates)} known duplicate pairs from QQP")

for i, (q1, q2) in enumerate(known_duplicates, 1):
    print(f"\n\nKnown duplicate pair {i}:")
    print(f"Q1: {q1}")
    print(f"Q2: {q2}")

    # Проверяем, что наша модель определяет их как дубликаты
    similarity = duplicate_finder.predict_similarity(q1, q2)
    print(f"Model similarity score: {similarity:.4f}")
    print(f"Prediction: {'DUPLICATE' if similarity >= 0.5 else 'NOT DUPLICATE'}")
    print(f"Correct: {'YES' if (similarity >= 0.5) == True else 'NO'}")



VERIFICATION WITH KNOWN QQP DUPLICATES
Found 5 known duplicate pairs from QQP


Known duplicate pair 1:
Q1: How do I control my horny emotions?
Q2: How do you control your horniness?
Model similarity score: 0.9736
Prediction: DUPLICATE
Correct: YES


Known duplicate pair 2:
Q1: What can one do after MBBS?
Q2: What do i do after my MBBS ?
Model similarity score: 0.9902
Prediction: DUPLICATE
Correct: YES


Known duplicate pair 3:
Q1: What is the best self help book you have read? Why? How did it change your life?
Q2: What are the top self help books I should read?
Model similarity score: 0.9349
Prediction: DUPLICATE
Correct: YES


Known duplicate pair 4:
Q1: What will be Hillary Clinton's policy towards India if she becomes president?
Q2: What will be Hilary Clinton's policy towards India if she become President?
Model similarity score: 0.9988
Prediction: DUPLICATE
Correct: YES


Known duplicate pair 5:
Q1: Which is the best book to study TENSOR for general relativity from basic?
Q2: W

In [ ]:
print("\n\n" + "="*80)
print("PERFORMANCE AND ACCURACY EVALUATION")
print("="*80)

def evaluate_search_accuracy(sample_size=100):
    print(f"Evaluating on {sample_size} samples...")

    correct_predictions = 0
    total_predictions = 0

    # Берем случайные пары из validation set
    indices = np.random.choice(len(eval_dataset), min(sample_size, len(eval_dataset)), replace=False)

    for idx in tqdm(indices, desc="Evaluating accuracy"):
        item = qqp["validation"][int(idx)]
        q1, q2, true_label = item["text1"], item["text2"], item["label"]

        # Предсказываем схожесть
        similarity = duplicate_finder.predict_similarity(q1, q2)
        predicted_label = 1 if similarity >= 0.5 else 0

        if predicted_label == true_label:
            correct_predictions += 1
        total_predictions += 1

    accuracy = correct_predictions / total_predictions
    print(f"Search accuracy: {accuracy:.4f} ({correct_predictions}/{total_predictions})")
    return accuracy

# Запускаем оценку
search_accuracy = evaluate_search_accuracy(100)
print(f"\nModel search accuracy on QQP validation set: {search_accuracy:.4f}")



PERFORMANCE AND ACCURACY EVALUATION
Evaluating on 100 samples...


Evaluating accuracy: 100%|██████████| 100/100 [00:08<00:00, 12.45it/s]

Search accuracy: 0.9700 (97/100)

Model search accuracy on QQP validation set: 0.9700


### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>